# Entrenamiento — ResNet50 con pesos de RadImageNet (variante: 6 clases, con exclusión de calidad)

Experimento aislado de transfer learning, no parte de la comparación de 10 arquitecturas: mismo modelo, mismos datos, misma receta de entrenamiento que el `ResNet50` ya entrenado con pesos de **ImageNet** — la única diferencia es el origen de los pesos preentrenados del backbone. Compara directo contra ese resultado ya guardado (`resultados_entrenamiento/6_clases_con_exclusion/resnet50/`).

**RadImageNet** (Mei et al., *Radiology: Artificial Intelligence*, 2022) es una red preentrenada con 1.35 millones de imágenes médicas (CT, MRI, ultrasonido — no fotos naturales como ImageNet). Solo publican pesos para 4 arquitecturas: ResNet50, DenseNet121, InceptionV3, InceptionResNetV2 — por eso este experimento usa ResNet50 (ya está en el roster con pesos ImageNet, comparación directa) y no se generaliza a los otros 9 modelos.

**Detalle importante, verificado antes de escribir este notebook** (no asumido): los pesos oficiales en PyTorch (enlace en el README de [BMEII-AI/RadImageNet](https://github.com/BMEII-AI/RadImageNet)) son el backbone de un `torchvision.resnet50` estándar (conv1...avgpool, sin `fc`) — se verificó cargándolos de verdad: `missing_keys` da exactamente `['fc.weight', 'fc.bias']` (la cabeza nueva) y `unexpected_keys` da vacío, o sea que calzan perfecto. El preprocesamiento **no es el mismo que usan los otros 10 modelos**: el propio `pytorch_example.ipynb` del repositorio oficial carga las imágenes con `cv2.imread` (canales **BGR**, no RGB) y normaliza con `(x-127.5)*2/255` (rango ~[-1,1], no la media/desviación estándar de ImageNet) — los pesos se calibraron con ese preprocesamiento exacto, así que este notebook lo reproduce en la Sección 2 (clase `BGRRescale`), en vez de reusar la normalización estándar del resto del roster.

**Antes de correr en Colab**: subir `dataset_final_224.zip` y `entrenamiento_utils.py` a la carpeta `BioMedInformatics` de Drive (igual que los demás notebooks) — los pesos de RadImageNet se descargan solos la primera vez (celda de la Sección 2, ~195MB, tarda un par de minutos).

## 0. Configuración

In [ ]:
import sys

RUNNING_IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = False  # True solo para una corrida de prueba rápida (pocas imágenes, 1+1 épocas)

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/BioMedInformatics"  # ajustar si la carpeta se llama distinto

if RUNNING_IN_COLAB:
    !pip install -q gdown
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/dataset_final_224
    !unzip -q -n {DRIVE_PROJECT_DIR}/dataset_final_224.zip -d /content/
    !cp {DRIVE_PROJECT_DIR}/entrenamiento_utils.py /content/
    sys.path.insert(0, "/content")
else:
    sys.path.insert(0, ".")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torchvision
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import entrenamiento_utils as tu

MODEL_NAME = "resnet50_radimagenet"
# Variante principal (la que se compara con el ResNet50 de ImageNet ya entrenado), no una de
# ablación -- se puede cambiar igual que en los demás notebooks si se quiere repetir el experimento
# en otra variante más adelante.
VARIANT_NAME = "6clases_con_exclusion"
CLASSES = tu.CLASSES_5 if "5clases" in VARIANT_NAME else tu.CLASSES_6

DATA_ROOT = Path("/content/dataset_final_224") if RUNNING_IN_COLAB else Path("../dataset_final_224")
MANIFEST_PATH = DATA_ROOT / "manifiestos" / f"manifiesto_{VARIANT_NAME}.csv"

# Los resultados van a Drive (persisten entre sesiones) cuando se corre en Colab -- DATA_ROOT
# sí puede vivir en el disco local porque el dataset se puede volver a descomprimir gratis,
# pero horas de entrenamiento no se pueden regenerar gratis si se pierden al cerrar la sesión.
# --- RÉPLICAS CON SEMILLA ----------------------------------------------------------------------
# SEED = None  -> corrida normal, igual que antes (sin semilla; resultados en resultados_entrenamiento/).
# SEED = 1, 2, 3... -> RÉPLICA: semilla fija (arranque reproducible) y los resultados van a
#                      resultados_entrenamiento/replicas/semilla_<N>/ para NO sobrescribir los originales.
# La semilla fija el arranque (pesos de la cabeza nueva, sampler, aumentos, orden de los batches); no
# garantiza igualdad bit a bit (cudnn.benchmark / operaciones atómicas de la GPU).
SEED = None
if SEED is not None:
    import random
    import numpy as np
    import torch
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
_BASE_RESULTS = Path(DRIVE_PROJECT_DIR) / "resultados_entrenamiento" if RUNNING_IN_COLAB else Path("../resultados/entrenamiento")
RESULTS_DIR = _BASE_RESULTS / "replicas" / f"semilla_{SEED}" if SEED is not None else _BASE_RESULTS
CHECKPOINT_PATH = RESULTS_DIR / f"{MODEL_NAME}__{VARIANT_NAME}_best.pt"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # entradas de tamaño fijo -> cuDNN elige el algoritmo más rápido

# Desactiva los kernels de atención "flash"/eficiente-en-memoria de PyTorch y deja solo el kernel
# clásico ("math", más lento pero sin este problema). Se agregó tras un error real entrenando
# MaxViT en Colab (RuntimeError: "LSE is not correctly aligned (strideH)") -- un bug conocido del
# kernel flash-attention con ciertas formas de tensor, no algo de nuestros datos ni de este código.
# Con las secuencias cortas de este proyecto (~200-260 tokens) el costo de rendimiento es mínimo.
if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

print(f"Dispositivo: {device} | RUNNING_IN_COLAB={RUNNING_IN_COLAB} | SMOKE_TEST={SMOKE_TEST}")
print(f"Resultados en: {RESULTS_DIR}")

## 1. Cargar el manifiesto de la variante

In [ ]:
splits = tu.load_manifest(MANIFEST_PATH, CLASSES)

if SMOKE_TEST:
    splits = {
        name: df.groupby("target_class").head(4).reset_index(drop=True)
        for name, df in splits.items()
    }

for name, df in splits.items():
    print(f"{name}: {len(df)} imágenes")
splits["train"]["target_class"].value_counts()

## 2. Crear el modelo, descargar los pesos de RadImageNet y las transformaciones de imagen
Este modelo no sale de `timm` (RadImageNet no está ahí) — se define aquí mismo (`RadImageNetResNet50`, misma arquitectura que `torchvision.models.resnet50`). El tamaño de entrada (224×224) es fijo, igual que el resto del roster; la normalización **sí es distinta** (ver `BGRRescale` en la celda de transformaciones) porque así se calibraron estos pesos.

In [ ]:
import zipfile

import gdown

PRETRAINED_CACHE_DIR = Path("/content/_pretrained_cache") if RUNNING_IN_COLAB else Path("../_pretrained_cache")
PRETRAINED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RADIMAGENET_ZIP = PRETRAINED_CACHE_DIR / "radimagenet_pytorch.zip"
RADIMAGENET_RESNET50_PT = PRETRAINED_CACHE_DIR / "RadImageNet_pytorch" / "ResNet50.pt"

# Pesos oficiales de RadImageNet (Mei et al., Radiology: AI 2022) -- enlace público listado en el
# README de https://github.com/BMEII-AI/RadImageNet, sección "Pretrained RadImageNet Models".
RADIMAGENET_GDRIVE_ID = "1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR"

# Respaldo: si ya subiste `radimagenet_pytorch.zip` a la carpeta BioMedInformatics de Drive, se usa ese archivo
# (evita el límite de cuota de descargas de Google Drive que a veces rompe gdown).
if RUNNING_IN_COLAB and not RADIMAGENET_ZIP.exists() and (Path(DRIVE_PROJECT_DIR) / "radimagenet_pytorch.zip").exists():
    import shutil
    shutil.copy(Path(DRIVE_PROJECT_DIR) / "radimagenet_pytorch.zip", RADIMAGENET_ZIP)
    print("Pesos de RadImageNet tomados de Drive (radimagenet_pytorch.zip).")

if not RADIMAGENET_RESNET50_PT.exists():
    if not RADIMAGENET_ZIP.exists():
        print("Descargando pesos de RadImageNet (~195MB, incluye ResNet50/DenseNet121/InceptionV3)...")
        gdown.download(url=f"https://drive.google.com/uc?id={RADIMAGENET_GDRIVE_ID}",
                       output=str(RADIMAGENET_ZIP), quiet=False)
    with zipfile.ZipFile(RADIMAGENET_ZIP) as z:
        z.extractall(PRETRAINED_CACHE_DIR)
print("Pesos de RadImageNet (ResNet50) en:", RADIMAGENET_RESNET50_PT, "| existe:", RADIMAGENET_RESNET50_PT.exists())


class RadImageNetResNet50(nn.Module):
    """ResNet50 estándar (misma arquitectura de torchvision que el ResNet50 con pesos ImageNet),
    con los pesos del backbone (hasta avgpool, sin fc) reemplazados por los de RadImageNet --
    entrenado solo con imágenes médicas (CT/MRI/ultrasonido), no con fotos naturales. La cabeza de
    clasificación (`fc`) es nueva, igual que en todos los demás modelos del roster."""

    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        base = torchvision.models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:9])  # conv1..avgpool, sin fc
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(2048, num_classes)
        if pretrained:
            state_dict = torch.load(RADIMAGENET_RESNET50_PT, map_location="cpu", weights_only=True)
            result = self.load_state_dict(state_dict, strict=False)
            assert result.missing_keys == ["fc.weight", "fc.bias"], result.missing_keys
            assert result.unexpected_keys == [], result.unexpected_keys
            print(f"Pesos de RadImageNet cargados en el backbone ({len(state_dict)} tensores) -- "
                  "solo 'fc.*' quedó sin inicializar (cabeza nueva).")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = self.flatten(x)
        return self.fc(x)

    def get_classifier(self) -> nn.Module:
        return self.fc


model = RadImageNetResNet50(num_classes=len(CLASSES), pretrained=True)
input_size = (224, 224)  # fijo -- igual que el resto del roster, no depende de resolve_data_config aquí
print("Tamaño de entrada:", input_size)
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,} "
      "(debería coincidir con el ResNet50 de ImageNet -- misma arquitectura, distinto origen de los pesos)")

In [ ]:
# Aumento de datos (solo train) -- MISMO esquema que en los otros 10 modelos: flip horizontal,
# rotación leve, zoom leve, ajuste de contraste leve. Lo único que cambia es el último paso: en vez
# de `transforms.Normalize` con media/desviación estándar de ImageNet, se usa `BGRRescale` -- los
# pesos de RadImageNet se calibraron con ESE preprocesamiento exacto (ver Sección 2 / intro).

class BGRRescale:
    """Reproduce el preprocesamiento oficial de RadImageNet (`pytorch_example.ipynb` del repositorio
    BMEII-AI/RadImageNet): reordena canales RGB->BGR y reescala de [0,1] a ~[-1,1] -- equivalente a
    `cv2.imread` (BGR) + `(x-127.5)*2/255` sobre la imagen en escala 0-255 original."""

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = tensor[[2, 1, 0], :, :]  # RGB -> BGR
        return tensor * 2.0 - 1.0  # [0,1] -> [-1,1]


train_transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.ColorJitter(contrast=0.1),
    transforms.ToTensor(),
    BGRRescale(),
])
eval_transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
    BGRRescale(),
])

## 3. Balanceo de clases: muestreo ponderado (no pesos de pérdida)

Silicosis (737 imágenes de train) y STB (529) tienen mucha menos data que las clases grandes (~4,200 cada una) — con solo pesos de pérdida, el modelo sigue *viendo* pocas imágenes de esas clases por época, solo se le da más peso al error cuando se equivoca en ellas. Un `WeightedRandomSampler` ataca el problema distinto: hace que el propio DataLoader muestree las clases raras con más frecuencia dentro de cada época (con reemplazo), así el modelo ve una mezcla más pareja de las 6 clases en cada batch.

**No se usan los dos mecanismos a la vez** (muestreo balanceado + pesos de pérdida) porque se sobre-compensaría el desbalance — se elige uno. Aquí se usa el sampler; la pérdida queda sin ponderar (`class_weights=None` en la Sección 6).

In [ ]:
conteo_train = splits["train"]["target_class"].value_counts()
print("Distribución real en train (antes de balancear el muestreo):")
print(conteo_train)

train_sampler = tu.make_weighted_sampler(splits["train"], CLASSES)

## 4. Datasets y DataLoaders

In [ ]:
BATCH_SIZE = 8 if SMOKE_TEST else 32
NUM_WORKERS = 0 if (RUNNING_IN_COLAB is False or SMOKE_TEST) else 2

train_ds = tu.CXRDataset(splits["train"], DATA_ROOT, CLASSES, transform=train_transform)
val_ds = tu.CXRDataset(splits["val"], DATA_ROOT, CLASSES, transform=eval_transform)
test_ds = tu.CXRDataset(splits["test"], DATA_ROOT, CLASSES, transform=eval_transform)

# pin_memory acelera la transferencia CPU->GPU; persistent_workers evita relanzar los
# workers en cada época (con NUM_WORKERS=0, como en local, ninguno de los dos hace nada)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                     persistent_workers=NUM_WORKERS > 0)

# shuffle=False porque el orden ya lo decide el sampler (son mutuamente excluyentes en PyTorch)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

# Verificación rápida: un batch real debe cargar sin error y tener la forma esperada
images, labels = next(iter(train_loader))
print("Forma de un batch:", images.shape, labels.shape)

In [ ]:
# Confirmar que el sampler de verdad está balanceando: contar clases en varios batches muestreados
from collections import Counter
conteo_muestreado = Counter()
for i, (_, labels) in enumerate(train_loader):
    conteo_muestreado.update(labels.tolist())
    if i >= 20:
        break
print("Distribución observada en ~20 batches muestreados (debería verse mucho más pareja que el conteo real):")
for i, c in enumerate(CLASSES):
    print(f"  {c}: {conteo_muestreado[i]}")

## 5. Entrenar en 2 fases

In [ ]:
PHASE1_EPOCHS = 1 if SMOKE_TEST else 8
PHASE2_EPOCHS = 1 if SMOKE_TEST else 12

model, history = tu.train_two_phase(
    model, train_loader, val_loader, class_weights=None, device=device,
    phase1_epochs=PHASE1_EPOCHS, phase1_lr=1e-3,
    phase2_epochs=PHASE2_EPOCHS, phase2_lr=1e-5,
    unfreeze_fraction=0.3, patience=5,
    checkpoint_path=CHECKPOINT_PATH,
)

In [ ]:
import pandas as pd
tu.plot_training_curves(history, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

## 6. Evaluar sobre el conjunto de test

In [ ]:
metrics = tu.evaluate_model(model, test_loader, CLASSES, device)
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Precision macro: {metrics['precision_macro']:.3f}")
print(f"Recall macro: {metrics['recall_macro']:.3f}")
print(f"F1 macro: {metrics['f1_macro']:.3f}")
print(f"AUC-ROC macro: {metrics['auc_roc_macro']:.3f}")
print(f"Tiempo de inferencia: {metrics['inference_time_ms_per_image']:.2f} ms/imagen")

In [ ]:
pd.DataFrame(metrics["per_class"])

In [ ]:
tu.plot_confusion_matrix(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.1 Curvas ROC por clase
Uno-contra-todos, con el AUC ya calculado en `evaluate_model`.

In [ ]:
tu.plot_roc_curves(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.2 Curvas Precision-Recall
Más informativa que ROC cuando el test está desbalanceado entre clases (Silicosis/STB tienen mucho menos soporte que Normal/Pneumonia/COVID-19) — ROC incluye los verdaderos negativos, que ahí dominan y pueden maquillar una precisión mala en la clase rara.

In [ ]:
tu.plot_pr_curves(metrics, CLASSES, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

### 6.3 Precision/Recall/F1 por clase
Mismos números de la tabla de la Sección 6, en forma de gráfico (barras de error = IC 95% de Wilson en precision/recall) — más fácil de comparar entre clases de un vistazo.

In [ ]:
tu.plot_per_class_metrics(metrics, MODEL_NAME, VARIANT_NAME, RESULTS_DIR)
plt.show()

## 7. Costo computacional

In [ ]:
params_info = tu.count_params(model)
size_mb = tu.model_size_mb(model)
print(f"Parámetros totales: {params_info['total_params']:,}")
print(f"Tamaño en disco: {size_mb:.2f} MB")
print(f"Tiempo de inferencia: {metrics['inference_time_ms_per_image']:.2f} ms/imagen")

## 8. Guardar resultados
Mismo formato para los 10 modelos, para el notebook de comparación final.

In [ ]:
out_path = tu.save_results(MODEL_NAME, VARIANT_NAME, history, metrics, params_info, size_mb, RESULTS_DIR, extra={"seed": SEED})
print(f"Resultados guardados en: {out_path}")
print(f"Métricas por clase (CSV): {RESULTS_DIR / f'{MODEL_NAME}__{VARIANT_NAME}_metricas_por_clase.csv'}")
print(f"Predicciones crudas (y_true/y_pred/y_prob, para pruebas de significancia u otras figuras "
      f"más adelante): {RESULTS_DIR / f'{MODEL_NAME}__{VARIANT_NAME}_predicciones.npz'}")
print(f"Tabla comparativa entre TODOS los modelos: {RESULTS_DIR / 'comparacion_modelos.csv'}")
print(f"Checkpoint del mejor modelo: {CHECKPOINT_PATH}")
print(f"Tiempo total de entrenamiento: {sum(history['epoch_time_s']) / 60:.1f} min")

## Notas
- `SMOKE_TEST = True` al inicio permite verificar que todo el pipeline corre de punta a punta (pocas imágenes, 1 época por fase) antes de lanzar el entrenamiento completo — útil para no gastar horas de GPU si hay un error de código.
- El balanceo de clases se hace con `WeightedRandomSampler` (Sección 3), no con pesos en la pérdida — por eso `train_two_phase` se llama con `class_weights=None`.
- Los resultados quedan en `resultados_entrenamiento/` dentro de la carpeta `BioMedInformatics` de Drive (persisten entre sesiones de Colab, no se pierden si la sesión se desconecta), listos para que el notebook de comparación final los junte con los de los otros 9 modelos.